# Fire VASE validation notebook

Run any section independently to reproduce one QA artifact. The default sample is real FIRED event `20657` with daily GridMET maximum temperature. Restore the materialized data lake under `data_lake/fire-vase-data-lake-v0.1/files` before running.

In [ ]:
from IPython.display import Image, display
from cubedynamics.validation import ValidationPaths

paths = ValidationPaths.discover()
FIRE_ID = 20657
VARIABLE = 'tmmx'

## 1. Pipe grammar and lazy backend

Compare the direct CubeDynamics verb with the pipe form over the same chunked GridMET subset.

In [ ]:
from cubedynamics.validation.pipeline import run_pipeline_validation

pipeline = run_pipeline_validation(paths, fire_id=FIRE_ID, variable=VARIABLE)
display(pipeline.metrics)
display(Image(filename=pipeline.artifacts['plot']))

## 2. FIRED polygon and hull sensitivity

Show the accepted 0–125 m simplification range alongside 500 m and 1000 m stress tests, then rebuild the directional time hull at each setting.

In [ ]:
from cubedynamics.validation.geometry import run_geometry_validation

geometry = run_geometry_validation(
    paths,
    fire_id=FIRE_ID,
    tolerances_m=(0, 125, 500, 1000),
    operational_max_tolerance_m=125,
    n_theta=96,
)
display(geometry.metrics)
display(Image(filename=geometry.artifacts['plot']))

## 3. GridMET date and polygon attribution

Recompute every lake-table centroid value from annual NetCDF and compare centroid, cell-center, and fractional pixel-overlap climate assignment.

In [ ]:
from cubedynamics.validation.climate import run_climate_validation

climate = run_climate_validation(paths, fire_id=FIRE_ID, variable=VARIABLE)
display(climate.metrics)
display(Image(filename=climate.artifacts['plot']))

## 4. External and upstream-source validation

The FIRED daily/event geometry check is offline. Set `EXTERNAL_NETWORK = True` to also query the independent NCAR/GDEX GridMET OPeNDAP mirror.

In [ ]:
from cubedynamics.validation.external import run_external_validation

EXTERNAL_NETWORK = False
external = run_external_validation(
    paths,
    fire_id=FIRE_ID,
    variable=VARIABLE,
    external_network=EXTERNAL_NETWORK,
)
display(external.metrics)
display(Image(filename=external.artifacts['plot']))

## Collate the current module results

The command-line runner normally creates the report. This cell shows the same collation step for results generated in this notebook.

In [ ]:
from cubedynamics.validation.report import build_validation_pdf

report = build_validation_pdf(
    [pipeline, geometry, climate, external],
    paths.repo_root / 'output/pdf/fire_vase_validation_report.pdf',
)
report